# Task O — AUG Sensitivity vs Full-Sample Benchmark Comparison

## Procedure 9 — Secondary Market

This notebook implements the final comparison required for the secondary-market analysis.

It compares:

- **Task L / Procedure 6:** full-sample in-sample optimization using the entire available AUG history as one hindsight benchmark;
- **Task M / Procedure 8:** rolling walk-forward sensitivity analysis under alternative in-sample window lengths \(T\) and out-of-sample horizons \(\tau\).

The purpose of Task O is **not** to perform another strategy optimization or to select a new walk-forward specification.

Instead, it evaluates how the rolling OOS results obtained under different \(T/\tau\) designs compare with the full-sample hindsight benchmark.

Following the Procedure 9 requirement, the comparison focuses on four dimensions:

1. **Return magnitude** — whether rolling OOS profitability materially deteriorates relative to the full-sample benchmark;
2. **Risk characteristics** — comparison of drawdown and risk-adjusted performance;
3. **Parameter stability** — whether the selected channel length and stop-loss parameter remain structurally consistent across walk-forward specifications;
4. **Performance sensitivity** — whether the strategy conclusions depend strongly on the selected \(T/\tau\) design.

Because Task L and Task M use different evaluation periods and fundamentally different parameter-selection procedures, their raw cumulative profits are not directly comparable.

Task L is a hindsight in-sample benchmark. Task M consists of rolling OOS sensitivity evaluations.

Accordingly, this notebook places greater emphasis on normalized performance measures, common-period Task M sensitivity ranges, parameter behavior, and qualitative evidence of performance deterioration or retention.

No new strategy specification is selected in Task O.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 220)
pd.set_option(
    "display.float_format",
    "{:,.6f}".format
)


# ============================================================
# 0.1 Finalized Task L
# Procedure 6 — Full-Sample Hindsight IS Benchmark
# ============================================================

TASK_L = {
    "Label":
        "Task L — Full-Sample IS Benchmark",

    "Procedure":
        6,

    "Evaluation_Type":
        "Full-Sample IS / Hindsight",

    "Sample_Start":
        pd.Timestamp("2018-05-03 09:05:00"),

    "Sample_End":
        pd.Timestamp("2026-04-10 15:00:00"),

    "Observations":
        138_744,

    "Starting_Equity":
        100_000.0,

    "Ending_Equity":
        1_515_095.395,

    "Net_Profit":
        1_415_095.395,

    "Total_Return":
        14.150954,

    "CAGR":
        0.408358,

    "Max_Drawdown_CNY":
        -16_520.4825,

    "Max_Drawdown_Pct":
        -0.069410,

    "Daily_Sharpe":
        3.687327,

    "Calmar":
        5.883255,

    "Completed_Trades":
        417,

    "Win_Rate":
        0.606715,

    "Profit_Factor":
        7.415238,

    "Expectancy":
        3_393.514137,

    "Payoff_Ratio":
        4.806716,

    "Optimal_ChnLen":
        710,

    "Optimal_StpPct":
        0.005,

    "StpPct_Lower_Bound":
        True,

    "ChnLen_Lower_Bound":
        False,

    "ChnLen_Upper_Bound":
        False,

    "Parameter_Evaluations":
        91_296,
}


# ============================================================
# 0.2 Finalized Task M
# Procedure 8 — Walk-Forward T / tau Sensitivity
# ============================================================

TASK_M = {
    "Label":
        "Task M — Walk-Forward Sensitivity",

    "Procedure":
        8,

    "Evaluation_Type":
        "Rolling OOS Sensitivity",

    "Specifications":
        6,

    "T_Years":
        (4, 5, 6),

    "Tau_Months":
        (3, 6),

    "Common_Start":
        pd.Timestamp("2024-07-01 09:05:00"),

    "Common_End":
        pd.Timestamp("2025-12-31 15:00:00"),

    "Common_Bars":
        26_496,

    "CAGR_Min":
        1.676813,

    "CAGR_Max":
        1.830535,

    "Sharpe_Min":
        3.979571,

    "Sharpe_Max":
        4.301115,

    "MDD_Pct_Min":
        -0.141416,

    "MDD_Pct_Max":
        -0.141416,

    "Calmar_Min":
        11.857294,

    "Calmar_Max":
        12.944313,

    "Median_ChnLen_Min":
        640.0,

    "Median_ChnLen_Max":
        1920.0,

    "StpPct_Lower_Bound_Share_Min":
        1.0,

    "StpPct_Lower_Bound_Share_Max":
        1.0,

    "Minimum_OOS_Windows":
        3,

    "Maximum_OOS_Windows":
        15,
}


# ============================================================
# 0.3 Finalized six-specification common-period results
# ============================================================

TASK_M_COMMON = pd.DataFrame(
    {
        "Specification": [
            "T4_tau3",
            "T4_tau6",
            "T5_tau3",
            "T5_tau6",
            "T6_tau3",
            "T6_tau6",
        ],

        "T_Years": [
            4, 4, 5, 5, 6, 6
        ],

        "Tau_Months": [
            3, 6, 3, 6, 3, 6
        ],

        "Full_Spec_OOS_Windows": [
            15, 7, 11, 5, 7, 3
        ],

        "Common_Period_CAGR": [
            1.676813,
            1.794759,
            1.676813,
            1.794759,
            1.698249,
            1.830535,
        ],

        "Common_Period_Sharpe": [
            3.979571,
            4.247695,
            3.979571,
            4.247695,
            4.025184,
            4.301115,
        ],

        "Common_Period_MDD_Pct": [
            -0.141416,
            -0.141416,
            -0.141416,
            -0.141416,
            -0.141416,
            -0.141416,
        ],

        "Common_Period_Calmar": [
            11.857294,
            12.691329,
            11.857294,
            12.691329,
            12.008872,
            12.944313,
        ],

        "Median_ChnLen": [
            1920.0,
            640.0,
            710.0,
            640.0,
            1920.0,
            1920.0,
        ],

        "StpPct_Lower_Bound_Share": [
            1.0,
            1.0,
            1.0,
            1.0,
            1.0,
            1.0,
        ],
    }
)


# ============================================================
# 0.4 Procedure mapping
# ============================================================

procedure_mapping = pd.DataFrame(
    {
        "Professor Procedure": [
            "Procedure 6",
            "Procedure 8",
            "Procedure 9",
        ],

        "Final Notebook": [
            "Task L",
            "Task M",
            "Task O",
        ],

        "Role": [
            "Full-sample hindsight IS benchmark",
            "Alternative T/tau rolling OOS sensitivity",
            "Procedure 8 vs Procedure 6 comparison",
        ],
    }
)


print("=" * 105)
print("TASK O — PROCEDURE MAPPING")
print("=" * 105)

display(procedure_mapping)


# ============================================================
# 0.5 Task L audit
# ============================================================

task_l_audit = pd.DataFrame(
    {
        "Item": [
            "Evaluation type",
            "Sample start",
            "Sample end",
            "Observations",
            "Parameter evaluations",
            "Optimal ChnLen",
            "Optimal StpPct",
            "Ending equity",
            "CAGR",
            "Daily Sharpe",
            "Maximum Drawdown (%)",
            "Calmar",
            "Profit Factor",
        ],

        "Result": [
            TASK_L["Evaluation_Type"],
            TASK_L["Sample_Start"],
            TASK_L["Sample_End"],
            TASK_L["Observations"],
            TASK_L["Parameter_Evaluations"],
            TASK_L["Optimal_ChnLen"],
            TASK_L["Optimal_StpPct"],
            TASK_L["Ending_Equity"],
            TASK_L["CAGR"],
            TASK_L["Daily_Sharpe"],
            TASK_L["Max_Drawdown_Pct"],
            TASK_L["Calmar"],
            TASK_L["Profit_Factor"],
        ],
    }
)


print()
print("=" * 105)
print("TASK L — FINALIZED PROCEDURE 6 INPUT")
print("=" * 105)

display(task_l_audit)


# ============================================================
# 0.6 Task M audit
# ============================================================

task_m_audit = pd.DataFrame(
    {
        "Item": [
            "Evaluation type",
            "Sensitivity specifications",
            "T values",
            "Tau values",
            "Common OOS start",
            "Common OOS end",
            "Common OOS bars",
            "Common-period CAGR minimum",
            "Common-period CAGR maximum",
            "Common-period Sharpe minimum",
            "Common-period Sharpe maximum",
            "Common-period MDD minimum",
            "Common-period MDD maximum",
            "Common-period Calmar minimum",
            "Common-period Calmar maximum",
        ],

        "Result": [
            TASK_M["Evaluation_Type"],
            TASK_M["Specifications"],
            TASK_M["T_Years"],
            TASK_M["Tau_Months"],
            TASK_M["Common_Start"],
            TASK_M["Common_End"],
            TASK_M["Common_Bars"],
            TASK_M["CAGR_Min"],
            TASK_M["CAGR_Max"],
            TASK_M["Sharpe_Min"],
            TASK_M["Sharpe_Max"],
            TASK_M["MDD_Pct_Min"],
            TASK_M["MDD_Pct_Max"],
            TASK_M["Calmar_Min"],
            TASK_M["Calmar_Max"],
        ],
    }
)


print()
print("=" * 105)
print("TASK M — FINALIZED PROCEDURE 8 INPUT")
print("=" * 105)

display(task_m_audit)


print()
print("=" * 105)
print("TASK M — SIX COMMON-PERIOD SPECIFICATIONS")
print("=" * 105)

display(TASK_M_COMMON)


# ============================================================
# 0.7 Hard consistency checks
# ============================================================

input_checks = pd.DataFrame(
    {
        "Validation Check": [
            "Task L correctly mapped to Procedure 6",
            "Task M correctly mapped to Procedure 8",
            "Task L identified as hindsight IS",
            "Task M identified as rolling OOS sensitivity",
            "Task L contains full 91,296-parameter search",
            "Task M contains six specifications",
            "Task M T values are 4, 5, and 6 years",
            "Task M tau values are 3 and 6 months",
            "Task M common period contains 26,496 bars",
            "Task M common table contains six rows",
            "All common-period CAGR values are positive",
            "All common-period Sharpe values are positive",
            "All common-period Calmar values are positive",
            "All Task M stop solutions remain at lower boundary",
            "Task L stop solution is at lower boundary",
        ],

        "Passed": [
            TASK_L["Procedure"] == 6,

            TASK_M["Procedure"] == 8,

            (
                TASK_L["Evaluation_Type"]
                ==
                "Full-Sample IS / Hindsight"
            ),

            (
                TASK_M["Evaluation_Type"]
                ==
                "Rolling OOS Sensitivity"
            ),

            TASK_L["Parameter_Evaluations"] == 91_296,

            TASK_M["Specifications"] == 6,

            tuple(TASK_M["T_Years"]) == (4, 5, 6),

            tuple(TASK_M["Tau_Months"]) == (3, 6),

            TASK_M["Common_Bars"] == 26_496,

            len(TASK_M_COMMON) == 6,

            (
                TASK_M_COMMON["Common_Period_CAGR"]
                > 0.0
            ).all(),

            (
                TASK_M_COMMON["Common_Period_Sharpe"]
                > 0.0
            ).all(),

            (
                TASK_M_COMMON["Common_Period_Calmar"]
                > 0.0
            ).all(),

            np.isclose(
                TASK_M_COMMON[
                    "StpPct_Lower_Bound_Share"
                ],
                1.0,
                rtol=0.0,
                atol=1e-12
            ).all(),

            TASK_L["StpPct_Lower_Bound"],
        ],
    }
)


print()
print("=" * 105)
print("SECTION 0 — FINALIZED INPUT VALIDATION")
print("=" * 105)

display(input_checks)

assert input_checks["Passed"].all()


# ============================================================
# 0.8 Interpretation
# ============================================================

print()
print("=" * 105)
print("SECTION 0 INTERPRETATION")
print("=" * 105)

print(
    "1. Task O compares finalized Procedure 8 rolling OOS "
    "sensitivity evidence with the finalized Procedure 6 "
    "full-sample hindsight IS benchmark."
)

print(
    "2. No new parameter optimization is performed in Task O."
)

print(
    "3. Task L and Task M use different evaluation designs and "
    "historical periods, so raw cumulative P&L is not treated "
    "as a directly comparable performance ranking."
)

print(
    "4. Task M common-period metrics provide the cleanest basis "
    "for evaluating sensitivity across alternative T/tau "
    "walk-forward specifications."
)

print(
    "5. Task O will focus on return magnitude, risk, parameter "
    "stability, and evidence of performance retention or decay."
)

print()
print("SECTION 0 VALIDATION PASSED.")

TASK O — PROCEDURE MAPPING


,Professor Procedure,Final Notebook,Role
0,Procedure 6,Task L,Full-sample hindsight IS benchmark
1,Procedure 8,Task M,Alternative T/tau rolling OOS sensitivity
2,Procedure 9,Task O,Procedure 8 vs Procedure 6 comparison



TASK L — FINALIZED PROCEDURE 6 INPUT


,Item,Result
0,Evaluation type,Full-Sample IS / Hindsight
1,Sample start,2018-05-03 09:05:00
2,Sample end,2026-04-10 15:00:00
3,Observations,138744
4,Parameter evaluations,91296
5,Optimal ChnLen,710
6,Optimal StpPct,0.005000
7,Ending equity,"1,515,095.395000"
8,CAGR,0.408358
9,Daily Sharpe,3.687327



TASK M — FINALIZED PROCEDURE 8 INPUT


...


TASK M — SIX COMMON-PERIOD SPECIFICATIONS


,Specification,T_Years,Tau_Months,Full_Spec_OOS_Windows,Common_Period_CAGR,Common_Period_Sharpe,Common_Period_MDD_Pct,Common_Period_Calmar,Median_ChnLen,StpPct_Lower_Bound_Share
0,T4_tau3,4,3,15,1.676813,3.979571,-0.141416,11.857294,"1,920.000000",1.000000
1,T4_tau6,4,6,7,1.794759,4.247695,-0.141416,12.691329,640.000000,1.000000
2,T5_tau3,5,3,11,1.676813,3.979571,-0.141416,11.857294,710.000000,1.000000
3,T5_tau6,5,6,5,1.794759,4.247695,-0.141416,12.691329,640.000000,1.000000
4,T6_tau3,6,3,7,1.698249,4.025184,-0.141416,12.008872,"1,920.000000",1.000000
5,T6_tau6,6,6,3,1.830535,4.301115,-0.141416,12.944313,"1,920.000000",1.000000



SECTION 0 — FINALIZED INPUT VALIDATION


,Validation Check,Passed
0,Task L correctly mapped to Procedure 6,True
1,Task M correctly mapped to Procedure 8,True
2,Task L identified as hindsight IS,True
3,Task M identified as rolling OOS sensitivity,True
4,"Task L contains full 91,296-parameter search",True
5,Task M contains six specifications,True
6,"Task M T values are 4, 5, and 6 years",True
7,Task M tau values are 3 and 6 months,True
8,"Task M common period contains 26,496 bars",True
9,Task M common table contains six rows,True



SECTION 0 INTERPRETATION
1. Task O compares finalized Procedure 8 rolling OOS sensitivity evidence with the finalized Procedure 6 full-sample hindsight IS benchmark.
2. No new parameter optimization is performed in Task O.
3. Task L and Task M use different evaluation designs and historical periods, so raw cumulative P&L is not treated as a directly comparable performance ranking.
4. Task M common-period metrics provide the cleanest basis for evaluating sensitivity across alternative T/tau walk-forward specifications.
5. Task O will focus on return magnitude, risk, parameter stability, and evidence of performance retention or decay.

SECTION 0 VALIDATION PASSED.


## 1. Return Magnitude and Risk-Adjusted Comparison

Procedure 9 requires an assessment of how the rolling OOS sensitivity results from Procedure 8 compare with the full-sample in-sample benchmark from Procedure 6.

A direct comparison of cumulative Net Profit is not appropriate because the two procedures cover different historical periods and use different parameter-selection designs.

Task L uses the entire available AUG history to select one globally optimal parameter pair and therefore represents a hindsight in-sample benchmark.

Task M evaluates alternative \(T/\tau\) walk-forward specifications out of sample. For sensitivity comparison across specifications, Task M uses an identical common OOS period from 2024-07-01 to 2025-12-31.

Accordingly, this section emphasizes normalized performance measures:

- CAGR,
- Daily Sharpe Ratio,
- Maximum Drawdown (%),
- Calmar Ratio.

The objective is not to rank Procedure 8 above or below Procedure 6 mechanically. Instead, the analysis asks whether the rolling OOS sensitivity results exhibit a clear collapse in profitability or risk-adjusted performance relative to the hindsight benchmark.

In [2]:
# ============================================================
# 1.1 Build normalized comparison table
# ============================================================

task_l_reference = {
    "CAGR": TASK_L["CAGR"],
    "Daily_Sharpe": TASK_L["Daily_Sharpe"],
    "Max_Drawdown_Pct": TASK_L["Max_Drawdown_Pct"],
    "Calmar": TASK_L["Calmar"],
}


comparison = TASK_M_COMMON[
    [
        "Specification",
        "T_Years",
        "Tau_Months",
        "Full_Spec_OOS_Windows",
        "Common_Period_CAGR",
        "Common_Period_Sharpe",
        "Common_Period_MDD_Pct",
        "Common_Period_Calmar",
    ]
].copy()


comparison["Task_L_CAGR"] = task_l_reference["CAGR"]

comparison["Task_L_Sharpe"] = (
    task_l_reference["Daily_Sharpe"]
)

comparison["Task_L_MDD_Pct"] = (
    task_l_reference["Max_Drawdown_Pct"]
)

comparison["Task_L_Calmar"] = (
    task_l_reference["Calmar"]
)


# ============================================================
# 1.2 Descriptive OOS / hindsight-IS ratios
#
# IMPORTANT:
# These are descriptive ratios only.
# Task M and Task L do not cover identical historical periods.
# ============================================================

comparison["CAGR_Ratio_vs_L"] = (
    comparison["Common_Period_CAGR"]
    /
    comparison["Task_L_CAGR"]
)

comparison["Sharpe_Ratio_vs_L"] = (
    comparison["Common_Period_Sharpe"]
    /
    comparison["Task_L_Sharpe"]
)

comparison["Absolute_MDD_Ratio_vs_L"] = (
    comparison["Common_Period_MDD_Pct"].abs()
    /
    abs(comparison["Task_L_MDD_Pct"])
)

comparison["Calmar_Ratio_vs_L"] = (
    comparison["Common_Period_Calmar"]
    /
    comparison["Task_L_Calmar"]
)


print("=" * 110)
print(
    "TASK O — NORMALIZED PROCEDURE 8 VS PROCEDURE 6 COMPARISON"
)
print("=" * 110)

display(
    comparison[
        [
            "Specification",
            "T_Years",
            "Tau_Months",
            "Full_Spec_OOS_Windows",
            "Common_Period_CAGR",
            "Task_L_CAGR",
            "Common_Period_Sharpe",
            "Task_L_Sharpe",
            "Common_Period_MDD_Pct",
            "Task_L_MDD_Pct",
            "Common_Period_Calmar",
            "Task_L_Calmar",
        ]
    ]
)


# ============================================================
# 1.3 Ratio summary
# ============================================================

ratio_table = comparison[
    [
        "Specification",
        "CAGR_Ratio_vs_L",
        "Sharpe_Ratio_vs_L",
        "Absolute_MDD_Ratio_vs_L",
        "Calmar_Ratio_vs_L",
    ]
].copy()


print()
print("=" * 110)
print("DESCRIPTIVE TASK M / TASK L RATIOS")
print("=" * 110)

display(ratio_table)


# ============================================================
# 1.4 Range summary
# ============================================================

range_summary = pd.DataFrame(
    {
        "Metric": [
            "CAGR",
            "Daily Sharpe",
            "Maximum Drawdown (%)",
            "Calmar",
        ],

        "Task L — Full-Sample IS": [
            TASK_L["CAGR"],
            TASK_L["Daily_Sharpe"],
            TASK_L["Max_Drawdown_Pct"],
            TASK_L["Calmar"],
        ],

        "Task M — Minimum": [
            comparison["Common_Period_CAGR"].min(),
            comparison["Common_Period_Sharpe"].min(),
            comparison["Common_Period_MDD_Pct"].min(),
            comparison["Common_Period_Calmar"].min(),
        ],

        "Task M — Maximum": [
            comparison["Common_Period_CAGR"].max(),
            comparison["Common_Period_Sharpe"].max(),
            comparison["Common_Period_MDD_Pct"].max(),
            comparison["Common_Period_Calmar"].max(),
        ],
    }
)


print()
print("=" * 110)
print("TASK L BENCHMARK VS TASK M COMMON-PERIOD RANGE")
print("=" * 110)

display(range_summary)


# ============================================================
# 1.5 Performance-retention / deterioration diagnostics
# ============================================================

min_cagr_ratio = float(
    comparison["CAGR_Ratio_vs_L"].min()
)

max_cagr_ratio = float(
    comparison["CAGR_Ratio_vs_L"].max()
)

min_sharpe_ratio = float(
    comparison["Sharpe_Ratio_vs_L"].min()
)

max_sharpe_ratio = float(
    comparison["Sharpe_Ratio_vs_L"].max()
)

min_calmar_ratio = float(
    comparison["Calmar_Ratio_vs_L"].min()
)

max_calmar_ratio = float(
    comparison["Calmar_Ratio_vs_L"].max()
)

min_abs_mdd_ratio = float(
    comparison["Absolute_MDD_Ratio_vs_L"].min()
)

max_abs_mdd_ratio = float(
    comparison["Absolute_MDD_Ratio_vs_L"].max()
)


performance_diagnostics = pd.DataFrame(
    {
        "Diagnostic": [
            "Minimum Task M CAGR / Task L CAGR",
            "Maximum Task M CAGR / Task L CAGR",
            "Minimum Task M Sharpe / Task L Sharpe",
            "Maximum Task M Sharpe / Task L Sharpe",
            "Minimum Task M Calmar / Task L Calmar",
            "Maximum Task M Calmar / Task L Calmar",
            "Minimum absolute MDD ratio vs Task L",
            "Maximum absolute MDD ratio vs Task L",
        ],

        "Value": [
            min_cagr_ratio,
            max_cagr_ratio,
            min_sharpe_ratio,
            max_sharpe_ratio,
            min_calmar_ratio,
            max_calmar_ratio,
            min_abs_mdd_ratio,
            max_abs_mdd_ratio,
        ],
    }
)


print()
print("=" * 110)
print("PERFORMANCE RETENTION / DETERIORATION DIAGNOSTICS")
print("=" * 110)

display(performance_diagnostics)


# ============================================================
# 1.6 Important period-design warning
# ============================================================

period_warning = pd.DataFrame(
    {
        "Evaluation": [
            "Task L — Procedure 6",
            "Task M — Procedure 8",
        ],

        "Period": [
            (
                f"{TASK_L['Sample_Start']} -> "
                f"{TASK_L['Sample_End']}"
            ),
            (
                f"{TASK_M['Common_Start']} -> "
                f"{TASK_M['Common_End']}"
            ),
        ],

        "Design": [
            "Full-sample hindsight IS optimization",
            "Common-period rolling OOS sensitivity",
        ],

        "Direct Raw P&L Comparison Valid?": [
            False,
            False,
        ],
    }
)


print()
print("=" * 110)
print("COMPARISON-DESIGN WARNING")
print("=" * 110)

display(period_warning)


# ============================================================
# 1.7 Hard validation
# ============================================================

section1_checks = pd.DataFrame(
    {
        "Validation Check": [
            "Six Task M specifications included",
            "All Task M common-period CAGR values positive",
            "All Task M common-period Sharpe values positive",
            "All Task M common-period Calmar values positive",
            "Task L CAGR positive",
            "Task L Sharpe positive",
            "Task L Calmar positive",
            "All CAGR ratios finite and positive",
            "All Sharpe ratios finite and positive",
            "All Calmar ratios finite and positive",
            "All absolute MDD ratios finite and positive",
            "Task M common-period MDD identical across six specs",
        ],

        "Passed": [
            len(comparison) == 6,

            (
                comparison["Common_Period_CAGR"]
                > 0.0
            ).all(),

            (
                comparison["Common_Period_Sharpe"]
                > 0.0
            ).all(),

            (
                comparison["Common_Period_Calmar"]
                > 0.0
            ).all(),

            TASK_L["CAGR"] > 0.0,

            TASK_L["Daily_Sharpe"] > 0.0,

            TASK_L["Calmar"] > 0.0,

            (
                np.isfinite(
                    comparison["CAGR_Ratio_vs_L"]
                )
                &
                (
                    comparison["CAGR_Ratio_vs_L"]
                    > 0.0
                )
            ).all(),

            (
                np.isfinite(
                    comparison["Sharpe_Ratio_vs_L"]
                )
                &
                (
                    comparison["Sharpe_Ratio_vs_L"]
                    > 0.0
                )
            ).all(),

            (
                np.isfinite(
                    comparison["Calmar_Ratio_vs_L"]
                )
                &
                (
                    comparison["Calmar_Ratio_vs_L"]
                    > 0.0
                )
            ).all(),

            (
                np.isfinite(
                    comparison[
                        "Absolute_MDD_Ratio_vs_L"
                    ]
                )
                &
                (
                    comparison[
                        "Absolute_MDD_Ratio_vs_L"
                    ]
                    > 0.0
                )
            ).all(),

            np.isclose(
                comparison[
                    "Common_Period_MDD_Pct"
                ].max(),
                comparison[
                    "Common_Period_MDD_Pct"
                ].min(),
                rtol=0.0,
                atol=1e-12
            ),
        ],
    }
)


print()
print("=" * 110)
print("SECTION 1 — VALIDATION")
print("=" * 110)

display(section1_checks)

assert section1_checks["Passed"].all()


# ============================================================
# 1.8 Interpretation
# ============================================================

print()
print("=" * 110)
print("SECTION 1 INTERPRETATION")
print("=" * 110)

print(
    "1. All six Task M walk-forward specifications remain "
    "profitable over the identical common OOS comparison period."
)

print(
    "2. The Task M common-period CAGR range is "
    f"{comparison['Common_Period_CAGR'].min():.2%} to "
    f"{comparison['Common_Period_CAGR'].max():.2%}, "
    "while the Task L full-sample hindsight CAGR is "
    f"{TASK_L['CAGR']:.2%}."
)

print(
    "3. The Task M common-period Daily Sharpe range is "
    f"{comparison['Common_Period_Sharpe'].min():.4f} to "
    f"{comparison['Common_Period_Sharpe'].max():.4f}, "
    "compared with "
    f"{TASK_L['Daily_Sharpe']:.4f} for Task L."
)

print(
    "4. All six Task M specifications experience the same "
    f"common-period maximum drawdown of "
    f"{comparison['Common_Period_MDD_Pct'].iloc[0]:.2%}, "
    "which is deeper than the Task L full-sample drawdown of "
    f"{TASK_L['Max_Drawdown_Pct']:.2%}."
)

print(
    "5. Task M common-period Calmar ratios remain strong, "
    f"ranging from "
    f"{comparison['Common_Period_Calmar'].min():.4f} to "
    f"{comparison['Common_Period_Calmar'].max():.4f}, "
    "compared with "
    f"{TASK_L['Calmar']:.4f} for Task L."
)

print(
    "6. On normalized return and risk-adjusted measures, the "
    "rolling OOS sensitivity results therefore do not exhibit "
    "a clear performance collapse relative to the hindsight "
    "benchmark."
)

print(
    "7. However, this should not be interpreted as evidence "
    "that Procedure 8 outperforms Procedure 6. The evaluations "
    "cover different historical periods, and Task L benefits "
    "from hindsight parameter selection."
)

print(
    "8. The deeper common-period OOS drawdown shows that strong "
    "return and Sharpe retention does not imply identical risk "
    "behavior."
)

print()
print("SECTION 1 VALIDATION PASSED.")

TASK O — NORMALIZED PROCEDURE 8 VS PROCEDURE 6 COMPARISON


,Specification,T_Years,Tau_Months,Full_Spec_OOS_Windows,Common_Period_CAGR,Task_L_CAGR,Common_Period_Sharpe,Task_L_Sharpe,Common_Period_MDD_Pct,Task_L_MDD_Pct,Common_Period_Calmar,Task_L_Calmar
0,T4_tau3,4,3,15,1.676813,0.408358,3.979571,3.687327,-0.141416,-0.069410,11.857294,5.883255
1,T4_tau6,4,6,7,1.794759,0.408358,4.247695,3.687327,-0.141416,-0.069410,12.691329,5.883255
2,T5_tau3,5,3,11,1.676813,0.408358,3.979571,3.687327,-0.141416,-0.069410,11.857294,5.883255
3,T5_tau6,5,6,5,1.794759,0.408358,4.247695,3.687327,-0.141416,-0.069410,12.691329,5.883255
4,T6_tau3,6,3,7,1.698249,0.408358,4.025184,3.687327,-0.141416,-0.069410,12.008872,5.883255
5,T6_tau6,6,6,3,1.830535,0.408358,4.301115,3.687327,-0.141416,-0.069410,12.944313,5.883255



DESCRIPTIVE TASK M / TASK L RATIOS


,Specification,CAGR_Ratio_vs_L,Sharpe_Ratio_vs_L,Absolute_MDD_Ratio_vs_L,Calmar_Ratio_vs_L
0,T4_tau3,4.106233,1.079256,2.037401,2.015431
1,T4_tau6,4.395063,1.151971,2.037401,2.157195
2,T5_tau3,4.106233,1.079256,2.037401,2.015431
3,T5_tau6,4.395063,1.151971,2.037401,2.157195
4,T6_tau3,4.158726,1.091627,2.037401,2.041195
5,T6_tau6,4.482672,1.166459,2.037401,2.200196



TASK L BENCHMARK VS TASK M COMMON-PERIOD RANGE


...


COMPARISON-DESIGN WARNING


,Evaluation,Period,Design,Direct Raw P&L Comparison Valid?
0,Task L — Procedure 6,2018-05-03 09:05:00 -> 2026-04-10 15:00:00,Full-sample hindsight IS optimization,False
1,Task M — Procedure 8,2024-07-01 09:05:00 -> 2025-12-31 15:00:00,Common-period rolling OOS sensitivity,False



SECTION 1 — VALIDATION


,Validation Check,Passed
0,Six Task M specifications included,True
1,All Task M common-period CAGR values positive,True
2,All Task M common-period Sharpe values positive,True
3,All Task M common-period Calmar values positive,True
4,Task L CAGR positive,True
5,Task L Sharpe positive,True
6,Task L Calmar positive,True
7,All CAGR ratios finite and positive,True
8,All Sharpe ratios finite and positive,True
9,All Calmar ratios finite and positive,True



SECTION 1 INTERPRETATION
1. All six Task M walk-forward specifications remain profitable over the identical common OOS comparison period.
2. The Task M common-period CAGR range is 167.68% to 183.05%, while the Task L full-sample hindsight CAGR is 40.84%.
3. The Task M common-period Daily Sharpe range is 3.9796 to 4.3011, compared with 3.6873 for Task L.
4. All six Task M specifications experience the same common-period maximum drawdown of -14.14%, which is deeper than the Task L full-sample drawdown of -6.94%.
5. Task M common-period Calmar ratios remain strong, ranging from 11.8573 to 12.9443, compared with 5.8833 for Task L.
6. On normalized return and risk-adjusted measures, the rolling OOS sensitivity results therefore do not exhibit a clear performance collapse relative to the hindsight benchmark.
7. However, this should not be interpreted as evidence that Procedure 8 outperforms Procedure 6. The evaluations cover different historical periods, and Task L benefits from hindsight p

## 2. Parameter Stability and Boundary Behavior

Procedure 9 also requires an assessment of parameter stability.

Task L identifies one globally optimal parameter pair using the entire AUG sample:

$$
ChnLen = 710, \qquad StpPct = 0.005.
$$

Task M, by contrast, repeatedly re-estimates the strategy parameters under six alternative walk-forward designs.

Parameter stability must therefore be interpreted carefully.

For `ChnLen`, variation across walk-forward specifications provides information about how sensitive the preferred breakout horizon is to the choice of training and testing windows.

For `StpPct`, apparent stability is complicated by the optimization boundary. The professor-specified grid begins at **0.005**, and every finalized Task M optimization selects this lower boundary. Task L also selects **0.005**.

Therefore, repeated selection of `StpPct = 0.005` should not be interpreted as evidence of a precisely identified interior optimum. Instead, it represents a persistent boundary solution: the experiment does not determine whether smaller stop percentages would improve the objective.

The prescribed parameter grid is retained unchanged. No ex-post grid expansion is performed in Task O.

In [3]:
# ============================================================
# 2.1 Parameter comparison table
# ============================================================

parameter_comparison = TASK_M_COMMON[
    [
        "Specification",
        "T_Years",
        "Tau_Months",
        "Full_Spec_OOS_Windows",
        "Median_ChnLen",
        "StpPct_Lower_Bound_Share",
    ]
].copy()


parameter_comparison["Task_L_Optimal_ChnLen"] = (
    TASK_L["Optimal_ChnLen"]
)

parameter_comparison["ChnLen_Difference_vs_L"] = (
    parameter_comparison["Median_ChnLen"]
    -
    TASK_L["Optimal_ChnLen"]
)

parameter_comparison["ChnLen_Ratio_vs_L"] = (
    parameter_comparison["Median_ChnLen"]
    /
    TASK_L["Optimal_ChnLen"]
)

parameter_comparison["Task_L_Optimal_StpPct"] = (
    TASK_L["Optimal_StpPct"]
)


print("=" * 110)
print("TASK O — PARAMETER STABILITY COMPARISON")
print("=" * 110)

display(parameter_comparison)


# ============================================================
# 2.2 Channel-length behavior
# ============================================================

chnlen_min = float(
    parameter_comparison["Median_ChnLen"].min()
)

chnlen_max = float(
    parameter_comparison["Median_ChnLen"].max()
)

chnlen_mean = float(
    parameter_comparison["Median_ChnLen"].mean()
)

chnlen_median = float(
    parameter_comparison["Median_ChnLen"].median()
)

chnlen_range = (
    chnlen_max
    -
    chnlen_min
)

chnlen_unique = int(
    parameter_comparison["Median_ChnLen"].nunique()
)


chnlen_summary = pd.DataFrame(
    {
        "Metric": [
            "Task L global optimal ChnLen",
            "Task M minimum median ChnLen",
            "Task M maximum median ChnLen",
            "Task M mean median ChnLen",
            "Task M median of median ChnLen",
            "Task M ChnLen range",
            "Unique Task M median ChnLen values",
        ],

        "Value": [
            TASK_L["Optimal_ChnLen"],
            chnlen_min,
            chnlen_max,
            chnlen_mean,
            chnlen_median,
            chnlen_range,
            chnlen_unique,
        ],
    }
)


print()
print("=" * 110)
print("CHANNEL-LENGTH STABILITY SUMMARY")
print("=" * 110)

display(chnlen_summary)


# ============================================================
# 2.3 Stop-percentage boundary behavior
# ============================================================

task_m_stop_min = float(
    parameter_comparison[
        "StpPct_Lower_Bound_Share"
    ].min()
)

task_m_stop_max = float(
    parameter_comparison[
        "StpPct_Lower_Bound_Share"
    ].max()
)

all_task_m_stop_boundary = bool(
    np.isclose(
        parameter_comparison[
            "StpPct_Lower_Bound_Share"
        ],
        1.0,
        rtol=0.0,
        atol=1e-12
    ).all()
)

task_l_stop_boundary = bool(
    TASK_L["StpPct_Lower_Bound"]
)


stop_boundary_summary = pd.DataFrame(
    {
        "Evidence": [
            "Professor grid lower bound",
            "Task L optimal StpPct",
            "Task L optimum at lower boundary",
            "Minimum Task M lower-bound frequency",
            "Maximum Task M lower-bound frequency",
            "All Task M specifications always at lower bound",
        ],

        "Value": [
            0.005,
            TASK_L["Optimal_StpPct"],
            task_l_stop_boundary,
            task_m_stop_min,
            task_m_stop_max,
            all_task_m_stop_boundary,
        ],
    }
)


print()
print("=" * 110)
print("STOP-PERCENTAGE BOUNDARY SUMMARY")
print("=" * 110)

display(stop_boundary_summary)


# ============================================================
# 2.4 Structural parameter diagnostics
# ============================================================

task_l_chnlen_inside_grid = (
    TASK_L["Optimal_ChnLen"] > 500
    and
    TASK_L["Optimal_ChnLen"] < 10_000
)

task_m_chnlen_varies = (
    chnlen_unique > 1
)

task_m_chnlen_contains_l_optimum = bool(
    np.isclose(
        parameter_comparison["Median_ChnLen"],
        TASK_L["Optimal_ChnLen"],
        rtol=0.0,
        atol=1e-12
    ).any()
)

persistent_stop_boundary = (
    task_l_stop_boundary
    and
    all_task_m_stop_boundary
)


structural_diagnostics = pd.DataFrame(
    {
        "Diagnostic": [
            "Task L ChnLen is interior to prescribed grid",
            "Task M median ChnLen varies across specifications",
            "Task M median ChnLen includes Task L optimum",
            "Task L StpPct is at prescribed lower boundary",
            "Task M StpPct is always at lower boundary",
            "Persistent StpPct boundary across L and M",
        ],

        "Result": [
            task_l_chnlen_inside_grid,
            task_m_chnlen_varies,
            task_m_chnlen_contains_l_optimum,
            task_l_stop_boundary,
            all_task_m_stop_boundary,
            persistent_stop_boundary,
        ],
    }
)


print()
print("=" * 110)
print("STRUCTURAL PARAMETER DIAGNOSTICS")
print("=" * 110)

display(structural_diagnostics)


# ============================================================
# 2.5 ChnLen descriptive distance from Task L
# ============================================================

chnlen_distance = parameter_comparison[
    [
        "Specification",
        "Median_ChnLen",
        "Task_L_Optimal_ChnLen",
        "ChnLen_Difference_vs_L",
        "ChnLen_Ratio_vs_L",
    ]
].copy()


print()
print("=" * 110)
print("CHANNEL-LENGTH DISTANCE FROM TASK L GLOBAL OPTIMUM")
print("=" * 110)

display(chnlen_distance)


# ============================================================
# 2.6 Hard validation
# ============================================================

section2_checks = pd.DataFrame(
    {
        "Validation Check": [
            "Six Task M specifications included",
            "Task L optimal ChnLen equals 710",
            "Task L optimal StpPct equals 0.005",
            "Task L ChnLen is not at lower grid boundary",
            "Task L ChnLen is not at upper grid boundary",
            "Task M median ChnLen varies across specifications",
            "Task M median ChnLen minimum equals 640",
            "Task M median ChnLen maximum equals 1920",
            "At least one Task M median ChnLen equals Task L optimum",
            "Task L StpPct is at lower boundary",
            "All Task M lower-bound frequencies equal 100%",
            "Persistent StpPct boundary identified",
        ],

        "Passed": [
            len(parameter_comparison) == 6,

            TASK_L["Optimal_ChnLen"] == 710,

            np.isclose(
                TASK_L["Optimal_StpPct"],
                0.005,
                rtol=0.0,
                atol=1e-12
            ),

            not TASK_L["ChnLen_Lower_Bound"],

            not TASK_L["ChnLen_Upper_Bound"],

            task_m_chnlen_varies,

            np.isclose(
                chnlen_min,
                640.0,
                rtol=0.0,
                atol=1e-12
            ),

            np.isclose(
                chnlen_max,
                1920.0,
                rtol=0.0,
                atol=1e-12
            ),

            task_m_chnlen_contains_l_optimum,

            task_l_stop_boundary,

            all_task_m_stop_boundary,

            persistent_stop_boundary,
        ],
    }
)


print()
print("=" * 110)
print("SECTION 2 — VALIDATION")
print("=" * 110)

display(section2_checks)

assert section2_checks["Passed"].all()


# ============================================================
# 2.7 Interpretation
# ============================================================

print()
print("=" * 110)
print("SECTION 2 INTERPRETATION")
print("=" * 110)

print(
    "1. Task L identifies an interior channel-length optimum "
    f"of ChnLen = {TASK_L['Optimal_ChnLen']}."
)

print(
    "2. Across the six Task M walk-forward specifications, "
    "the median selected ChnLen ranges from "
    f"{chnlen_min:.0f} to {chnlen_max:.0f}."
)

print(
    "3. The Task M channel-length summaries therefore show "
    "variation across T/tau designs rather than complete "
    "parameter invariance."
)

print(
    "4. At least one Task M specification has a median "
    "ChnLen equal to the Task L global optimum of 710, "
    "while other specifications favor longer channel lengths."
)

print(
    "5. StpPct behaves differently: Task L selects 0.005, "
    "and every Task M specification reports a 100% "
    "lower-bound selection frequency."
)

print(
    "6. This repeated StpPct selection should not be interpreted "
    "as a precisely identified stable optimum because 0.005 is "
    "the minimum admissible value in the prescribed grid."
)

print(
    "7. The experiment therefore does not determine whether "
    "smaller StpPct values would further improve the objective."
)

print(
    "8. Overall, the AUG evidence shows moderate sensitivity "
    "in the preferred channel horizon together with a persistent "
    "stop-parameter boundary solution."
)

print()
print("SECTION 2 VALIDATION PASSED.")

TASK O — PARAMETER STABILITY COMPARISON


,Specification,T_Years,Tau_Months,Full_Spec_OOS_Windows,Median_ChnLen,StpPct_Lower_Bound_Share,Task_L_Optimal_ChnLen,ChnLen_Difference_vs_L,ChnLen_Ratio_vs_L,Task_L_Optimal_StpPct
0,T4_tau3,4,3,15,"1,920.000000",1.000000,710,"1,210.000000",2.704225,0.005000
1,T4_tau6,4,6,7,640.000000,1.000000,710,-70.000000,0.901408,0.005000
2,T5_tau3,5,3,11,710.000000,1.000000,710,0.000000,1.000000,0.005000
3,T5_tau6,5,6,5,640.000000,1.000000,710,-70.000000,0.901408,0.005000
4,T6_tau3,6,3,7,"1,920.000000",1.000000,710,"1,210.000000",2.704225,0.005000
5,T6_tau6,6,6,3,"1,920.000000",1.000000,710,"1,210.000000",2.704225,0.005000



CHANNEL-LENGTH STABILITY SUMMARY


,Metric,Value
0,Task L global optimal ChnLen,710.000000
1,Task M minimum median ChnLen,640.000000
2,Task M maximum median ChnLen,"1,920.000000"
3,Task M mean median ChnLen,"1,291.666667"
4,Task M median of median ChnLen,"1,315.000000"
5,Task M ChnLen range,"1,280.000000"
6,Unique Task M median ChnLen values,3.000000



STOP-PERCENTAGE BOUNDARY SUMMARY


...


CHANNEL-LENGTH DISTANCE FROM TASK L GLOBAL OPTIMUM


,Specification,Median_ChnLen,Task_L_Optimal_ChnLen,ChnLen_Difference_vs_L,ChnLen_Ratio_vs_L
0,T4_tau3,"1,920.000000",710,"1,210.000000",2.704225
1,T4_tau6,640.000000,710,-70.000000,0.901408
2,T5_tau3,710.000000,710,0.000000,1.000000
3,T5_tau6,640.000000,710,-70.000000,0.901408
4,T6_tau3,"1,920.000000",710,"1,210.000000",2.704225
5,T6_tau6,"1,920.000000",710,"1,210.000000",2.704225



SECTION 2 — VALIDATION


,Validation Check,Passed
0,Six Task M specifications included,True
1,Task L optimal ChnLen equals 710,True
2,Task L optimal StpPct equals 0.005,True
3,Task L ChnLen is not at lower grid boundary,True
4,Task L ChnLen is not at upper grid boundary,True
5,Task M median ChnLen varies across specifications,True
6,Task M median ChnLen minimum equals 640,True
7,Task M median ChnLen maximum equals 1920,True
8,At least one Task M median ChnLen equals Task ...,True
9,Task L StpPct is at lower boundary,True



SECTION 2 INTERPRETATION
1. Task L identifies an interior channel-length optimum of ChnLen = 710.
2. Across the six Task M walk-forward specifications, the median selected ChnLen ranges from 640 to 1920.
3. The Task M channel-length summaries therefore show variation across T/tau designs rather than complete parameter invariance.
4. At least one Task M specification has a median ChnLen equal to the Task L global optimum of 710, while other specifications favor longer channel lengths.
5. StpPct behaves differently: Task L selects 0.005, and every Task M specification reports a 100% lower-bound selection frequency.
6. This repeated StpPct selection should not be interpreted as a precisely identified stable optimum because 0.005 is the minimum admissible value in the prescribed grid.
7. The experiment therefore does not determine whether smaller StpPct values would further improve the objective.
8. Overall, the AUG evidence shows moderate sensitivity in the preferred channel horizon toge

## 3. Performance Sensitivity and Decay Assessment

The final objective of Procedure 9 is to determine whether the rolling OOS results are highly sensitive to the selected walk-forward design and whether there is evidence of substantial performance deterioration relative to the full-sample hindsight benchmark.

Two distinct questions must be separated.

First, **walk-forward sensitivity** asks whether the Procedure 8 conclusions change materially when \(T\) and \(\tau\) are altered.

Second, **performance decay** asks whether the rolling OOS evidence shows a clear deterioration relative to the Procedure 6 hindsight benchmark.

The Task M sensitivity analysis provides the cleanest evidence for the first question because all six specifications are evaluated over an identical common OOS period.

The comparison with Task L must be interpreted more cautiously because Task L covers a longer historical period and uses full-sample hindsight optimization.

Therefore, evidence of performance retention or decay is evaluated using normalized metrics and qualitative consistency rather than raw cumulative P&L.

No specification is selected as a new “best” walk-forward design in this section.

In [4]:
# ============================================================
# 3.1 Task M sensitivity ranges
# ============================================================

sensitivity_summary = pd.DataFrame(
    {
        "Metric": [
            "Common-period CAGR",
            "Common-period Daily Sharpe",
            "Common-period Maximum Drawdown (%)",
            "Common-period Calmar",
            "Median ChnLen",
            "Full-spec OOS window count",
        ],

        "Minimum": [
            TASK_M_COMMON["Common_Period_CAGR"].min(),
            TASK_M_COMMON["Common_Period_Sharpe"].min(),
            TASK_M_COMMON["Common_Period_MDD_Pct"].min(),
            TASK_M_COMMON["Common_Period_Calmar"].min(),
            TASK_M_COMMON["Median_ChnLen"].min(),
            TASK_M_COMMON["Full_Spec_OOS_Windows"].min(),
        ],

        "Maximum": [
            TASK_M_COMMON["Common_Period_CAGR"].max(),
            TASK_M_COMMON["Common_Period_Sharpe"].max(),
            TASK_M_COMMON["Common_Period_MDD_Pct"].max(),
            TASK_M_COMMON["Common_Period_Calmar"].max(),
            TASK_M_COMMON["Median_ChnLen"].max(),
            TASK_M_COMMON["Full_Spec_OOS_Windows"].max(),
        ],
    }
)


sensitivity_summary["Range"] = (
    sensitivity_summary["Maximum"]
    -
    sensitivity_summary["Minimum"]
)


print("=" * 110)
print("TASK O — WALK-FORWARD SENSITIVITY RANGE SUMMARY")
print("=" * 110)

display(sensitivity_summary)


# ============================================================
# 3.2 Relative dispersion across specifications
# ============================================================

cagr_min = float(
    TASK_M_COMMON["Common_Period_CAGR"].min()
)

cagr_max = float(
    TASK_M_COMMON["Common_Period_CAGR"].max()
)

sharpe_min = float(
    TASK_M_COMMON["Common_Period_Sharpe"].min()
)

sharpe_max = float(
    TASK_M_COMMON["Common_Period_Sharpe"].max()
)

calmar_min = float(
    TASK_M_COMMON["Common_Period_Calmar"].min()
)

calmar_max = float(
    TASK_M_COMMON["Common_Period_Calmar"].max()
)


cagr_relative_range = (
    (cagr_max - cagr_min)
    /
    cagr_min
)

sharpe_relative_range = (
    (sharpe_max - sharpe_min)
    /
    sharpe_min
)

calmar_relative_range = (
    (calmar_max - calmar_min)
    /
    calmar_min
)


dispersion_summary = pd.DataFrame(
    {
        "Metric": [
            "CAGR",
            "Daily Sharpe",
            "Calmar",
        ],

        "Minimum": [
            cagr_min,
            sharpe_min,
            calmar_min,
        ],

        "Maximum": [
            cagr_max,
            sharpe_max,
            calmar_max,
        ],

        "Absolute Range": [
            cagr_max - cagr_min,
            sharpe_max - sharpe_min,
            calmar_max - calmar_min,
        ],

        "Relative Range vs Minimum": [
            cagr_relative_range,
            sharpe_relative_range,
            calmar_relative_range,
        ],
    }
)


print()
print("=" * 110)
print("RELATIVE DISPERSION ACROSS TASK M SPECIFICATIONS")
print("=" * 110)

display(dispersion_summary)


# ============================================================
# 3.3 Profitability consistency
# ============================================================

all_cagr_positive = bool(
    (
        TASK_M_COMMON["Common_Period_CAGR"]
        >
        0.0
    ).all()
)

all_sharpe_positive = bool(
    (
        TASK_M_COMMON["Common_Period_Sharpe"]
        >
        0.0
    ).all()
)

all_calmar_positive = bool(
    (
        TASK_M_COMMON["Common_Period_Calmar"]
        >
        0.0
    ).all()
)

identical_common_mdd = bool(
    np.isclose(
        TASK_M_COMMON[
            "Common_Period_MDD_Pct"
        ].max(),
        TASK_M_COMMON[
            "Common_Period_MDD_Pct"
        ].min(),
        rtol=0.0,
        atol=1e-12
    )
)


profitability_consistency = pd.DataFrame(
    {
        "Diagnostic": [
            "All six common-period CAGR values positive",
            "All six common-period Sharpe values positive",
            "All six common-period Calmar values positive",
            "Common-period MDD identical across six specs",
        ],

        "Result": [
            all_cagr_positive,
            all_sharpe_positive,
            all_calmar_positive,
            identical_common_mdd,
        ],
    }
)


print()
print("=" * 110)
print("TASK M PROFITABILITY CONSISTENCY")
print("=" * 110)

display(profitability_consistency)


# ============================================================
# 3.4 Procedure 8 vs Procedure 6 decay diagnostics
# ============================================================

min_task_m_cagr_vs_l = (
    cagr_min
    /
    TASK_L["CAGR"]
)

min_task_m_sharpe_vs_l = (
    sharpe_min
    /
    TASK_L["Daily_Sharpe"]
)

min_task_m_calmar_vs_l = (
    calmar_min
    /
    TASK_L["Calmar"]
)

task_m_mdd_vs_l = (
    abs(
        TASK_M_COMMON[
            "Common_Period_MDD_Pct"
        ].iloc[0]
    )
    /
    abs(
        TASK_L["Max_Drawdown_Pct"]
    )
)


# These thresholds are descriptive diagnostics only.
# They are NOT statistical hypothesis tests.

no_clear_cagr_collapse = (
    min_task_m_cagr_vs_l > 0.90
)

no_clear_sharpe_collapse = (
    min_task_m_sharpe_vs_l > 0.90
)

no_clear_calmar_collapse = (
    min_task_m_calmar_vs_l > 0.90
)

deeper_oos_drawdown = (
    task_m_mdd_vs_l > 1.0
)


decay_diagnostics = pd.DataFrame(
    {
        "Diagnostic": [
            "Minimum Task M CAGR / Task L CAGR",
            "Minimum Task M Sharpe / Task L Sharpe",
            "Minimum Task M Calmar / Task L Calmar",
            "Task M absolute MDD / Task L absolute MDD",
            "No clear CAGR collapse",
            "No clear Sharpe collapse",
            "No clear Calmar collapse",
            "Task M common-period drawdown deeper than Task L",
        ],

        "Value": [
            min_task_m_cagr_vs_l,
            min_task_m_sharpe_vs_l,
            min_task_m_calmar_vs_l,
            task_m_mdd_vs_l,
            no_clear_cagr_collapse,
            no_clear_sharpe_collapse,
            no_clear_calmar_collapse,
            deeper_oos_drawdown,
        ],
    }
)


print()
print("=" * 110)
print("PROCEDURE 8 VS PROCEDURE 6 DECAY DIAGNOSTICS")
print("=" * 110)

display(decay_diagnostics)


# ============================================================
# 3.5 Sample-length limitation
# ============================================================

min_oos_windows = int(
    TASK_M_COMMON[
        "Full_Spec_OOS_Windows"
    ].min()
)

max_oos_windows = int(
    TASK_M_COMMON[
        "Full_Spec_OOS_Windows"
    ].max()
)

limited_long_window_evidence = (
    min_oos_windows <= 3
)


window_count_summary = TASK_M_COMMON[
    [
        "Specification",
        "T_Years",
        "Tau_Months",
        "Full_Spec_OOS_Windows",
    ]
].copy()


print()
print("=" * 110)
print("OOS WINDOW-COUNT LIMITATION")
print("=" * 110)

display(window_count_summary)

print()
print(
    f"Minimum full-spec OOS windows : {min_oos_windows}"
)

print(
    f"Maximum full-spec OOS windows : {max_oos_windows}"
)

print(
    "Limited evidence for longest specifications : "
    f"{limited_long_window_evidence}"
)


# ============================================================
# 3.6 Final sensitivity / decay classification
# ============================================================

performance_consistent_across_specs = (
    all_cagr_positive
    and
    all_sharpe_positive
    and
    all_calmar_positive
)

normalized_performance_no_collapse = (
    no_clear_cagr_collapse
    and
    no_clear_sharpe_collapse
    and
    no_clear_calmar_collapse
)


assessment_flags = pd.DataFrame(
    {
        "Assessment Dimension": [
            "Profitability consistent across tested T/tau designs",
            "Risk-adjusted performance consistent across tested designs",
            "Normalized performance shows no clear collapse vs Task L",
            "OOS common-period drawdown is deeper than Task L",
            "Channel horizon varies across walk-forward designs",
            "Stop parameter remains a persistent lower-bound solution",
            "Longest sensitivity specifications have limited OOS windows",
        ],

        "Flag": [
            all_cagr_positive,
            performance_consistent_across_specs,
            normalized_performance_no_collapse,
            deeper_oos_drawdown,
            task_m_chnlen_varies,
            persistent_stop_boundary,
            limited_long_window_evidence,
        ],
    }
)


print()
print("=" * 110)
print("TASK O — PERFORMANCE SENSITIVITY / DECAY FLAGS")
print("=" * 110)

display(assessment_flags)


# ============================================================
# 3.7 Hard validation
# ============================================================

section3_checks = pd.DataFrame(
    {
        "Validation Check": [
            "Six sensitivity specifications included",
            "All common-period CAGR values positive",
            "All common-period Sharpe values positive",
            "All common-period Calmar values positive",
            "CAGR relative range finite",
            "Sharpe relative range finite",
            "Calmar relative range finite",
            "Common-period MDD identical across six specs",
            "No clear Sharpe collapse vs Task L",
            "No clear Calmar collapse vs Task L",
            "Task M common-period drawdown deeper than Task L",
            "Minimum OOS window count equals 3",
            "Maximum OOS window count equals 15",
            "Limited longest-window evidence identified",
            "Persistent stop-boundary limitation retained",
        ],

        "Passed": [
            len(TASK_M_COMMON) == 6,

            all_cagr_positive,

            all_sharpe_positive,

            all_calmar_positive,

            np.isfinite(
                cagr_relative_range
            ),

            np.isfinite(
                sharpe_relative_range
            ),

            np.isfinite(
                calmar_relative_range
            ),

            identical_common_mdd,

            no_clear_sharpe_collapse,

            no_clear_calmar_collapse,

            deeper_oos_drawdown,

            min_oos_windows == 3,

            max_oos_windows == 15,

            limited_long_window_evidence,

            persistent_stop_boundary,
        ],
    }
)


print()
print("=" * 110)
print("SECTION 3 — VALIDATION")
print("=" * 110)

display(section3_checks)

assert section3_checks["Passed"].all()


# ============================================================
# 3.8 Interpretation
# ============================================================

print()
print("=" * 110)
print("SECTION 3 INTERPRETATION")
print("=" * 110)

print(
    "1. All six Task M specifications remain profitable and "
    "retain positive risk-adjusted performance over the identical "
    "common OOS comparison period."
)

print(
    "2. The common-period CAGR varies from "
    f"{cagr_min:.2%} to {cagr_max:.2%}, "
    f"a relative range of {cagr_relative_range:.2%}."
)

print(
    "3. Daily Sharpe varies from "
    f"{sharpe_min:.4f} to {sharpe_max:.4f}, "
    f"a relative range of {sharpe_relative_range:.2%}."
)

print(
    "4. Calmar varies from "
    f"{calmar_min:.4f} to {calmar_max:.4f}, "
    f"a relative range of {calmar_relative_range:.2%}."
)

print(
    "5. The six specifications therefore show limited sensitivity "
    "in headline normalized performance within the tested "
    "T/tau range."
)

print(
    "6. Relative to Task L, the Task M normalized CAGR, Sharpe, "
    "and Calmar measures do not exhibit a clear historical "
    "performance collapse."
)

print(
    "7. However, the common-period Task M maximum drawdown is "
    "materially deeper than the Task L full-sample drawdown, "
    "so risk behavior is not invariant across the evaluations."
)

print(
    "8. Parameter stability is also incomplete: the preferred "
    "channel horizon changes across specifications, while the "
    "stop parameter remains pinned to the prescribed lower grid "
    "boundary."
)

print(
    "9. The longest Task M specifications contain as few as "
    f"{min_oos_windows} OOS windows, limiting the strength of "
    "statistical inference."
)

print(
    "10. Overall, Procedure 8 does not show clear performance "
    "decay within the tested historical range, but this should "
    "not be interpreted as proof of live robustness or as "
    "evidence that rolling OOS performance is superior to the "
    "full-sample hindsight benchmark."
)

print()
print("SECTION 3 VALIDATION PASSED.")

TASK O — WALK-FORWARD SENSITIVITY RANGE SUMMARY


,Metric,Minimum,Maximum,Range
0,Common-period CAGR,1.676813,1.830535,0.153722
1,Common-period Daily Sharpe,3.979571,4.301115,0.321544
2,Common-period Maximum Drawdown (%),-0.141416,-0.141416,0.000000
3,Common-period Calmar,11.857294,12.944313,1.087019
4,Median ChnLen,640.000000,"1,920.000000","1,280.000000"
5,Full-spec OOS window count,3.000000,15.000000,12.000000



RELATIVE DISPERSION ACROSS TASK M SPECIFICATIONS


,Metric,Minimum,Maximum,Absolute Range,Relative Range vs Minimum
0,CAGR,1.676813,1.830535,0.153722,0.091675
1,Daily Sharpe,3.979571,4.301115,0.321544,0.080799
2,Calmar,11.857294,12.944313,1.087019,0.091675



TASK M PROFITABILITY CONSISTENCY


...


Minimum full-spec OOS windows : 3
Maximum full-spec OOS windows : 15
Limited evidence for longest specifications : True

TASK O — PERFORMANCE SENSITIVITY / DECAY FLAGS


,Assessment Dimension,Flag
0,Profitability consistent across tested T/tau d...,True
1,Risk-adjusted performance consistent across te...,True
2,Normalized performance shows no clear collapse...,True
3,OOS common-period drawdown is deeper than Task L,True
4,Channel horizon varies across walk-forward des...,True
5,Stop parameter remains a persistent lower-boun...,True
6,Longest sensitivity specifications have limite...,True



SECTION 3 — VALIDATION


,Validation Check,Passed
0,Six sensitivity specifications included,True
1,All common-period CAGR values positive,True
2,All common-period Sharpe values positive,True
3,All common-period Calmar values positive,True
4,CAGR relative range finite,True
5,Sharpe relative range finite,True
6,Calmar relative range finite,True
7,Common-period MDD identical across six specs,True
8,No clear Sharpe collapse vs Task L,True
9,No clear Calmar collapse vs Task L,True



SECTION 3 INTERPRETATION
1. All six Task M specifications remain profitable and retain positive risk-adjusted performance over the identical common OOS comparison period.
2. The common-period CAGR varies from 167.68% to 183.05%, a relative range of 9.17%.
3. Daily Sharpe varies from 3.9796 to 4.3011, a relative range of 8.08%.
4. Calmar varies from 11.8573 to 12.9443, a relative range of 9.17%.
5. The six specifications therefore show limited sensitivity in headline normalized performance within the tested T/tau range.
6. Relative to Task L, the Task M normalized CAGR, Sharpe, and Calmar measures do not exhibit a clear historical performance collapse.
7. However, the common-period Task M maximum drawdown is materially deeper than the Task L full-sample drawdown, so risk behavior is not invariant across the evaluations.
8. Parameter stability is also incomplete: the preferred channel horizon changes across specifications, while the stop parameter remains pinned to the prescribed lower 

## 4. Final Procedure 9 Comparison and Conclusion

Procedure 9 requires a final comparison between:

- **Procedure 6 / Task L:** the full-sample hindsight in-sample benchmark;
- **Procedure 8 / Task M:** rolling OOS sensitivity analysis under alternative \(T/\tau\) designs.

The comparison is organized around four dimensions:

1. return magnitude;
2. risk characteristics;
3. parameter stability;
4. performance decay.

The objective is not to select a new “best” \(T/\tau\) specification.

Instead, the purpose is to determine whether the strong AUG results remain structurally consistent when the walk-forward design changes, and how this evidence should be interpreted relative to the idealized full-sample benchmark.

Because the two procedures use different historical periods and parameter-selection mechanisms, raw cumulative profits are descriptive only. The final assessment therefore relies primarily on normalized performance, common-period sensitivity evidence, parameter behavior, and identified limitations.

In [5]:
# ============================================================
# 4.1 Consolidated evidence
# ============================================================

final_evidence = pd.DataFrame(
    {
        "Dimension": [
            "Return magnitude",
            "Risk-adjusted performance",
            "Maximum drawdown",
            "Walk-forward sensitivity",
            "Channel-length stability",
            "Stop-parameter behavior",
            "Sample-length limitation",
            "Performance decay assessment",
        ],

        "Evidence": [
            (
                f"Task M common-period CAGR = "
                f"{100 * cagr_min:.2f}% to "
                f"{100 * cagr_max:.2f}%; "
                f"Task L full-sample CAGR = "
                f"{100 * TASK_L['CAGR']:.2f}%"
            ),

            (
                f"Task M Sharpe = "
                f"{sharpe_min:.4f} to "
                f"{sharpe_max:.4f}; "
                f"Task L Sharpe = "
                f"{TASK_L['Daily_Sharpe']:.4f}"
            ),

            (
                f"Task M common-period MDD = "
                f"{100 * TASK_M_COMMON['Common_Period_MDD_Pct'].iloc[0]:.2f}%; "
                f"Task L full-sample MDD = "
                f"{100 * TASK_L['Max_Drawdown_Pct']:.2f}%"
            ),

            (
                f"Six T/tau specifications; "
                f"CAGR relative range = "
                f"{100 * cagr_relative_range:.2f}%; "
                f"Sharpe relative range = "
                f"{100 * sharpe_relative_range:.2f}%"
            ),

            (
                f"Task L ChnLen = "
                f"{TASK_L['Optimal_ChnLen']}; "
                f"Task M median ChnLen range = "
                f"{chnlen_min:.0f} to "
                f"{chnlen_max:.0f}"
            ),

            (
                f"Task L StpPct = "
                f"{TASK_L['Optimal_StpPct']:.3f}; "
                f"Task M lower-bound frequency = 100%"
            ),

            (
                f"Task M full-spec OOS windows range from "
                f"{min_oos_windows} to "
                f"{max_oos_windows}"
            ),

            (
                "No clear collapse in normalized return or "
                "risk-adjusted performance within the tested "
                "historical range, but OOS drawdown is deeper."
            ),
        ],

        "Assessment": [
            "Strong historical OOS return evidence",
            "No clear risk-adjusted performance collapse",
            "OOS risk is materially less favorable",
            "Limited sensitivity within tested window designs",
            "Moderate channel-horizon variation",
            "Persistent lower-bound parameter limitation",
            "Longest specifications have limited evidence",
            "Historical robustness evidence, not proof of live robustness",
        ],
    }
)


print("=" * 115)
print("TASK O — FINAL PROCEDURE 9 EVIDENCE")
print("=" * 115)

display(final_evidence)


# ============================================================
# 4.2 Final quantitative summary
# ============================================================

final_quantitative_summary = pd.DataFrame(
    {
        "Metric": [
            "Task L CAGR",
            "Task M common-period CAGR minimum",
            "Task M common-period CAGR maximum",
            "Task L Daily Sharpe",
            "Task M common-period Sharpe minimum",
            "Task M common-period Sharpe maximum",
            "Task L Maximum Drawdown (%)",
            "Task M common-period Maximum Drawdown (%)",
            "Task L Calmar",
            "Task M common-period Calmar minimum",
            "Task M common-period Calmar maximum",
            "Task L optimal ChnLen",
            "Task M minimum median ChnLen",
            "Task M maximum median ChnLen",
            "Task L optimal StpPct",
            "Task M StpPct lower-bound frequency minimum",
            "Task M StpPct lower-bound frequency maximum",
            "Minimum Task M OOS window count",
            "Maximum Task M OOS window count",
        ],

        "Value": [
            TASK_L["CAGR"],
            cagr_min,
            cagr_max,
            TASK_L["Daily_Sharpe"],
            sharpe_min,
            sharpe_max,
            TASK_L["Max_Drawdown_Pct"],
            TASK_M_COMMON["Common_Period_MDD_Pct"].iloc[0],
            TASK_L["Calmar"],
            calmar_min,
            calmar_max,
            TASK_L["Optimal_ChnLen"],
            chnlen_min,
            chnlen_max,
            TASK_L["Optimal_StpPct"],
            task_m_stop_min,
            task_m_stop_max,
            min_oos_windows,
            max_oos_windows,
        ],
    }
)


print()
print("=" * 115)
print("FINAL QUANTITATIVE SUMMARY")
print("=" * 115)

display(final_quantitative_summary)


# ============================================================
# 4.3 Final Procedure 9 assessment flags
# ============================================================

final_assessment_flags = pd.DataFrame(
    {
        "Assessment": [
            "All six Task M specifications profitable on common OOS period",
            "All six Task M specifications have positive Sharpe",
            "All six Task M specifications have positive Calmar",
            "Headline normalized performance has limited sensitivity",
            "No clear normalized performance collapse vs Task L",
            "Task M common-period drawdown deeper than Task L",
            "Task M ChnLen varies across specifications",
            "Task L ChnLen remains inside prescribed grid",
            "Task L StpPct at prescribed lower boundary",
            "Task M StpPct always at prescribed lower boundary",
            "Longest Task M specifications have limited OOS windows",
            "Raw cumulative P&L comparison should not be treated as direct ranking",
        ],

        "Flag": [
            all_cagr_positive,
            all_sharpe_positive,
            all_calmar_positive,

            (
                cagr_relative_range < 0.15
                and
                sharpe_relative_range < 0.15
                and
                calmar_relative_range < 0.15
            ),

            normalized_performance_no_collapse,

            deeper_oos_drawdown,

            task_m_chnlen_varies,

            task_l_chnlen_inside_grid,

            task_l_stop_boundary,

            all_task_m_stop_boundary,

            limited_long_window_evidence,

            True,
        ],
    }
)


print()
print("=" * 115)
print("FINAL PROCEDURE 9 ASSESSMENT FLAGS")
print("=" * 115)

display(final_assessment_flags)


# ============================================================
# 4.4 Final validation
# ============================================================

final_checks = pd.DataFrame(
    {
        "Validation Check": [
            "Procedure 6 benchmark correctly represented by Task L",
            "Procedure 8 sensitivity correctly represented by Task M",
            "Six Task M specifications included",
            "All Task M common-period CAGR values positive",
            "All Task M common-period Sharpe values positive",
            "All Task M common-period Calmar values positive",
            "Task M headline metric dispersion remains below 15%",
            "Task M drawdown deeper than Task L identified",
            "Task M ChnLen variation identified",
            "Task L ChnLen interior-grid status preserved",
            "Task L stop-boundary limitation preserved",
            "Task M persistent stop-boundary limitation preserved",
            "Minimum OOS window count equals 3",
            "Maximum OOS window count equals 15",
            "No new T/tau specification selected",
            "Different-period comparison limitation retained",
        ],

        "Passed": [
            TASK_L["Procedure"] == 6,

            TASK_M["Procedure"] == 8,

            len(TASK_M_COMMON) == 6,

            all_cagr_positive,

            all_sharpe_positive,

            all_calmar_positive,

            (
                cagr_relative_range < 0.15
                and
                sharpe_relative_range < 0.15
                and
                calmar_relative_range < 0.15
            ),

            deeper_oos_drawdown,

            task_m_chnlen_varies,

            task_l_chnlen_inside_grid,

            task_l_stop_boundary,

            all_task_m_stop_boundary,

            min_oos_windows == 3,

            max_oos_windows == 15,

            True,

            True,
        ],
    }
)


print()
print("=" * 115)
print("TASK O — FINAL VALIDATION")
print("=" * 115)

display(final_checks)

assert final_checks["Passed"].all()


# ============================================================
# 4.5 Final interpretation
# ============================================================

print()
print("=" * 115)
print("FINAL PROCEDURE 9 ASSESSMENT")
print("=" * 115)

print(
    "1. Procedure 8 remains historically profitable across all "
    "six tested T/tau walk-forward designs over the identical "
    "common OOS comparison period."
)

print(
    "2. Headline normalized performance is relatively stable "
    "across the tested specifications: CAGR, Sharpe, and Calmar "
    "vary within comparatively narrow ranges."
)

print(
    "3. Relative to the Procedure 6 full-sample hindsight "
    "benchmark, the Procedure 8 normalized performance measures "
    "do not exhibit a clear historical collapse."
)

print(
    "4. This does not mean that Procedure 8 outperforms "
    "Procedure 6. The two evaluations cover different historical "
    "periods and use fundamentally different parameter-selection "
    "designs."
)

print(
    "5. Risk characteristics are not identical. The Procedure 8 "
    "common-period maximum drawdown is materially deeper than "
    "the Procedure 6 full-sample drawdown."
)

print(
    "6. Parameter stability is mixed. The preferred ChnLen varies "
    "across walk-forward designs, indicating moderate sensitivity "
    "in the breakout horizon."
)

print(
    "7. StpPct = 0.005 is selected at the prescribed lower grid "
    "boundary in Task L and throughout Task M. This is a persistent "
    "boundary solution rather than evidence of a precisely "
    "identified unconstrained optimum."
)

print(
    "8. The longest Task M specifications contain only a small "
    "number of OOS windows, so conclusions about window-design "
    "robustness remain constrained by the relatively short AUG "
    "history."
)

print(
    "9. Taken together, Procedure 9 provides evidence that the "
    "AUG strategy's historical OOS performance is not highly "
    "sensitive to the tested T/tau designs and does not show a "
    "clear normalized performance collapse relative to the "
    "hindsight benchmark."
)

print(
    "10. The evidence nevertheless should not be interpreted as "
    "proof of deployable live performance, parameter invariance, "
    "or the absence of overfitting, regime dependence, or "
    "data-path bias."
)

print()
print("ALL TASK O FINAL VALIDATION CHECKS PASSED.")

TASK O — FINAL PROCEDURE 9 EVIDENCE


,Dimension,Evidence,Assessment
0,Return magnitude,Task M common-period CAGR = 167.68% to 183.05%...,Strong historical OOS return evidence
1,Risk-adjusted performance,Task M Sharpe = 3.9796 to 4.3011; Task L Sharp...,No clear risk-adjusted performance collapse
2,Maximum drawdown,Task M common-period MDD = -14.14%; Task L ful...,OOS risk is materially less favorable
3,Walk-forward sensitivity,Six T/tau specifications; CAGR relative range ...,Limited sensitivity within tested window designs
4,Channel-length stability,Task L ChnLen = 710; Task M median ChnLen rang...,Moderate channel-horizon variation
5,Stop-parameter behavior,Task L StpPct = 0.005; Task M lower-bound freq...,Persistent lower-bound parameter limitation
6,Sample-length limitation,Task M full-spec OOS windows range from 3 to 15,Longest specifications have limited evidence
7,Performance decay assessment,No clear collapse in normalized return or risk...,"Historical robustness evidence, not proof of l..."



FINAL QUANTITATIVE SUMMARY


,Metric,Value
0,Task L CAGR,0.408358
1,Task M common-period CAGR minimum,1.676813
2,Task M common-period CAGR maximum,1.830535
3,Task L Daily Sharpe,3.687327
4,Task M common-period Sharpe minimum,3.979571
5,Task M common-period Sharpe maximum,4.301115
6,Task L Maximum Drawdown (%),-0.069410
7,Task M common-period Maximum Drawdown (%),-0.141416
8,Task L Calmar,5.883255
9,Task M common-period Calmar minimum,11.857294



FINAL PROCEDURE 9 ASSESSMENT FLAGS


,Assessment,Flag
0,All six Task M specifications profitable on co...,True
1,All six Task M specifications have positive Sh...,True
2,All six Task M specifications have positive Ca...,True
3,Headline normalized performance has limited se...,True
4,No clear normalized performance collapse vs Ta...,True
5,Task M common-period drawdown deeper than Task L,True
6,Task M ChnLen varies across specifications,True
7,Task L ChnLen remains inside prescribed grid,True
8,Task L StpPct at prescribed lower boundary,True
9,Task M StpPct always at prescribed lower boundary,True



TASK O — FINAL VALIDATION


,Validation Check,Passed
0,Procedure 6 benchmark correctly represented by...,True
1,Procedure 8 sensitivity correctly represented ...,True
2,Six Task M specifications included,True
3,All Task M common-period CAGR values positive,True
4,All Task M common-period Sharpe values positive,True
5,All Task M common-period Calmar values positive,True
6,Task M headline metric dispersion remains belo...,True
7,Task M drawdown deeper than Task L identified,True
8,Task M ChnLen variation identified,True
9,Task L ChnLen interior-grid status preserved,True



FINAL PROCEDURE 9 ASSESSMENT
1. Procedure 8 remains historically profitable across all six tested T/tau walk-forward designs over the identical common OOS comparison period.
2. Headline normalized performance is relatively stable across the tested specifications: CAGR, Sharpe, and Calmar vary within comparatively narrow ranges.
3. Relative to the Procedure 6 full-sample hindsight benchmark, the Procedure 8 normalized performance measures do not exhibit a clear historical collapse.
4. This does not mean that Procedure 8 outperforms Procedure 6. The two evaluations cover different historical periods and use fundamentally different parameter-selection designs.
5. Risk characteristics are not identical. The Procedure 8 common-period maximum drawdown is materially deeper than the Procedure 6 full-sample drawdown.
6. Parameter stability is mixed. The preferred ChnLen varies across walk-forward designs, indicating moderate sensitivity in the breakout horizon.
7. StpPct = 0.005 is selected at